In [ ]:
import h5py
import pandas as pd
import numpy as np
import os
from pathlib import Path
from matplotlib import pyplot as plt
import scipy.stats
from decoding_utils import apply_condition_filter
import vbn_utils
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
from notebook_utils import (
    image_name_dict,
    calcDprime,
    calcHitRate,
    get_contingent_engaged_trials,
    beh_mat_from_stim_table,
    mean_beh_mat_across_sessions,
    get_omission_response_rate,
    get_post_omission_response_rate,
    get_shared_hit_rate,
    get_private_hit_rate,
    get_shared_fa_rate,
    get_private_fa_rate,
    get_shared_nonchange_response_rate,
    get_private_nonchange_response_rate,
    skip_diag_masking,
    get_session_engaged_dprime,
    get_session_engaged_hit_count,
    get_experience_session_id_for_mouse,
    comparison_matrix,
    calculate_metric_for_selection,
    paired_image_mat_from_stim_table,
    mean_paired_image_mat_from_stim_table
)

In [ ]:
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['pdf.fonttype'] = 42

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"
unit_table_with_rf_stats = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_rf_stats.csv"
unit_table_opto_metrics = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_opto_metrics.csv"
unit_cluster_labels = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_cluster_labels.csv"
unit_vis_responsive = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_vis_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv"

In [ ]:
units = pd.read_csv(unit_table_file)
unnamedcols = [c for c in units.columns if 'Unnamed' in c]
units = units.drop(columns=unnamedcols)

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

sessions_table = pd.read_csv(sessions_table_file)

## Add behavioral metrics to sessions table

In [ ]:
#Add dprime and hit count to sessions table
dprimes = []
hit_counts = []
for _, session in sessions_table.iterrows():

    session_id = session['ecephys_session_id']
    dprime = get_session_engaged_dprime(stim_table, session_id)
    hit_count = get_session_engaged_hit_count(stim_table, session_id)

    dprimes.append(dprime)
    hit_counts.append(hit_count)


sessions_table['engaged_dprime'] = dprimes
sessions_table['engaged_hitcount'] = hit_counts

In [ ]:
#Add timing or visual strategy label to sessions table
behavior_model_summary = pd.read_pickle("/Volumes/programs/braintv/workgroups/nc-ophys/alex.piet/NP/behavior/psy_fits_v100/summary_data/_summary_table.pkl")
timing_sessions = behavior_model_summary[behavior_model_summary['strategy_labels']=='timing'].index.values
visual_sessions = behavior_model_summary[behavior_model_summary['strategy_labels']=='visual'].index.values

#sessions_table.set_index('ecephys_session_id', inplace=True)
sessions_table['strategy'] = 'visual'
sessions_table.loc[timing_sessions, 'strategy'] = 'timing'

In [ ]:
sessions_table.to_csv(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables", 'master_sessions_table.csv'))

## Add derived columns to stim table

In [ ]:
stim_table['engaged'] = stim_table['reward_rate']>=2

In [ ]:
stim_table = stim_table.merge(sessions_table[['experience_level', 'ecephys_session_id']], left_on='session_id', right_on='ecephys_session_id')

In [ ]:
lick_bouts = [False]*len(stim_table)
last_lick_time = -2
last_session = 0
for iflash, flash in stim_table.iterrows():
    sessionid = flash['session_id']
    if not np.isnan(flash['lick_time']):
        if (flash['lick_time'] - last_lick_time>2) or sessionid != last_session:
            lick_bouts[iflash] = True
            last_lick_time = flash['lick_time']
            last_session = sessionid

stim_table['lick_bout_start'] = lick_bouts

In [ ]:
stim_table['lick_for_flash_during_response_window'] = stim_table['lick_for_flash'] & \
                                                        (stim_table['lick_time']-stim_table['start_time'] > 0.1) & \
                                                        (stim_table['lick_time']-stim_table['start_time'] < 0.75) 

stim_table['lickbout_for_flash_during_response_window'] = stim_table['lick_bout_start'] & \
                                                        (stim_table['lick_time']-stim_table['start_time'] > 0.1) & \
                                                        (stim_table['lick_time']-stim_table['start_time'] < 0.75) 

In [ ]:
stim_table['grace_period_after_hit'] = False
stim_table.loc[stim_table['hit'] & \
                (stim_table['flashes_since_change']>0) & \
                (stim_table['change_frame'].notna()) & \
                (stim_table['start_frame']>stim_table['change_frame']),'grace_period_after_hit'] = True

In [ ]:
no_abnorm_sessions = sessions_table[sessions_table['abnormal_activity'].isnull() & sessions_table['abnormal_histology'].isnull()]['ecephys_session_id']

stim_table['no_abnorm'] = stim_table['session_id'].isin(no_abnorm_sessions)

In [ ]:
#Add is_sham_change column to stim_trials
def get_is_sham_change(stim_table_session_group):
    catch_frames = stim_table_session_group[stim_table_session_group['catch']==True]['change_frame'].unique()
    stim_table_session_group['is_sham_change'] = False
    catch_flashes = stim_table_session_group[stim_table_session_group['start_frame'].isin(catch_frames)].index.values
    stim_table_session_group.loc[catch_flashes, 'is_sham_change'] = True
    stim_table_session_group.loc[stim_table_session_group.stimulus_block==5, 'is_sham_change'] = stim_table_session_group[stim_table_session_group.active]['is_sham_change'].values #copy to the passive block
    return stim_table_session_group['is_sham_change']
    
sham_changes = stim_table.groupby('session_id').apply(get_is_sham_change)
stim_table['is_sham_change'] = sham_changes.values
stim_table['change_eligible_window'] = (stim_table['trial_flash']>=4)&(~stim_table['grace_period_after_hit'])

In [ ]:
change_inds = stim_table[stim_table['is_change']].index.values
stim_table['is_prechange'] = False
stim_table.loc[change_inds-1, 'is_prechange'] = True

In [ ]:
stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
stim_table.to_csv(stim_table_file)

## Session-level behavioral metrics

In [ ]:
good_sessions = sessions_table[sessions_table['abnormal_histology'].isnull() & sessions_table['abnormal_activity'].isnull()]
good_behavior_sessions = good_sessions[(good_sessions['engaged_dprime']>=1)&(good_sessions['engaged_hitcount']>=50)]

In [ ]:
fig, ax = plt.subplots()
ax.hist(good_sessions['engaged_hitcount'], color='k')
ax.set_xlabel('Number of engaged rewards per session')
ax.set_ylabel('Session count')

In [ ]:
#good_sessions['engaged_dprime'].hist()
fig, ax = plt.subplots()
ax.hist(good_sessions['engaged_dprime'], color='k')
ax.set_xlabel('Engaged d-prime')
ax.set_ylabel('Session count')

## Hit rates for holdover images on Familiar and Novel days

In [ ]:
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['pdf.fonttype'] = 42

In [ ]:
import warnings
warnings.filterwarnings("ignore")

for training_trajectory in ['GGH', 'GHG', 'HHG', 'all']:
    beh_mat_dict = {'Familiar':[], 'Novel':[]}
    image_set_dict = {}
    for experience_level in ['Familiar', 'Novel']:
        sessions_to_analyze = good_sessions[apply_condition_filter(good_sessions, experience_level, training_trajectory)]['ecephys_session_id'].unique()

        ims, counts, beh_mats = mean_beh_mat_across_sessions(stim_table, sessions_to_analyze)
        beh_mat_dict[experience_level] = beh_mats
        image_set_dict[experience_level] = ims[0]

        
    fig, axes = plt.subplots(1,3)
    fig.set_size_inches(10,5)
    fig.suptitle(f'{training_trajectory}: {len(sessions_to_analyze)} sessions')
    for experience_level, ax in zip(['Familiar', 'Novel'], axes[:2]):
        beh_mats = beh_mat_dict[experience_level]
        im = ax.imshow(np.nanmean(beh_mats, axis=0), clim=[0,1])
        ax.set_xlabel('Change Image')
        ax.set_ylabel('Pre-change Image')
        ax.set_xticks(np.arange(8))
        ax.set_xticklabels(image_set_dict[experience_level], rotation=90)
        ax.set_yticks(np.arange(8))
        ax.set_yticklabels(image_set_dict[experience_level])
        # if experience_level=='Familiar':
        #     plt.colorbar(im)    

    holdover_responses_83 = {'Familiar':[], 'Novel':[]}
    holdover_responses_111 = {'Familiar':[], 'Novel':[]}
    for experience_level in ['Familiar', 'Novel']: 
        beh_mats = beh_mat_dict[experience_level]
        beh_mats = np.array([skip_diag_masking(b) for b in beh_mats])
        holdover_responses_83[experience_level].append(np.nanmean(beh_mats[:, :, 6], axis = 0))
        holdover_responses_111[experience_level].append(np.nanmean(beh_mats[:, :, 7], axis = 0))

    axes[2].plot(holdover_responses_83['Familiar'], holdover_responses_83['Novel'], 'ko')
    axes[2].plot(holdover_responses_111['Familiar'], holdover_responses_111['Novel'], 'ko', markerfacecolor='w', markeredgewidth=2)
    axes[2].plot([0,1], [0,1], 'k--')
    axes[2].set_aspect('equal')
    axes[2].set_xlabel('Hit rate during familiar sessions')
    axes[2].set_ylabel('Hit rate during novel sessions')


    
    plt.tight_layout()
    fig.savefig(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript", f'behavioral_response_matrix_{training_trajectory}_test.png'))

In [ ]:
mouse_session_counts = good_sessions.value_counts('mouse_id')
mice_with_fam_and_nov_sessions = mouse_session_counts[mouse_session_counts > 1].index.values
mice_with_fam_and_nov_sessions


In [ ]:
sessions_to_analyze = good_sessions['ecephys_session_id'].unique()

session_beh_mats = {}
beh_mat_array = []
count_array = []
session_labels = []
experience_labels = []
image_set_labels = []
for _, session in good_sessions.iterrows():
    session_id = session['ecephys_session_id']
    mouse_id = session['mouse_id']
    experience = session['experience_level']

    ims, counts, beh_mat = beh_mat_from_stim_table(stim_table, session_id)
    image_set = 'G' if 'im036_r' in ims else 'H'
    
    beh_mat_array.append(beh_mat)
    count_array.append(counts)
    session_labels.append(session_id)
    experience_labels.append(experience)
    image_set_labels.append(image_set)


    session_beh_mats[session_id] = {'beh_mat': beh_mat, 'counts': counts, 'images': ims}
    

In [ ]:
beh_mat_array = np.array(beh_mat_array)
beh_mat_no_diag_array = np.array([skip_diag_masking(b) for b in beh_mat_array])

count_array = np.array(count_array)
count_no_diag_array = np.array([skip_diag_masking(b) for b in count_array])

good_behavior_session_filter = np.array([(get_session_engaged_dprime(stim_table, session_id)>1) & \
                                (get_session_engaged_hit_count(stim_table, session_id)>50) for session_id in session_labels])

experience_labels = np.array(experience_labels)
session_labels = np.array(session_labels)
image_set_labels = np.array(image_set_labels)


## Hit rates on holdover images for Novel days for novel/familiar pre-images

In [ ]:
im_83_hit_rates = {'Familiar':[], 'Novel':[], 'Familiar_from_holdover':[], 'Novel_from_holdover':[], }
im_111_hit_rates = {'Familiar':[], 'Novel':[], 'Familiar_from_holdover':[], 'Novel_from_holdover':[], }

for mouse in mice_with_fam_and_nov_sessions:

    for experience in ['Familiar', 'Novel']:

        session_id = get_experience_session_id_for_mouse(sessions_table, mouse, experience)

        beh_mat = session_beh_mats[session_id]['beh_mat']
        counts = session_beh_mats[session_id]['counts']
        ims = session_beh_mats[session_id]['images']

        beh_mat_no_diag = skip_diag_masking(beh_mat)
        counts_no_diag = skip_diag_masking(counts)


        for ind, imrates in zip([6,7], [im_83_hit_rates[experience], im_111_hit_rates[experience]]):

            ind_count = np.nansum(counts_no_diag[:, ind])
            if (ind_count > 10) & (get_session_engaged_dprime(stim_table, session_id)>1) & (get_session_engaged_hit_count(stim_table, session_id)>50):

                #print(ind_count, get_session_engaged_dprime(stim_table, session_id), get_session_engaged_hit_count(stim_table, session_id))
                imrates.append(np.nanmean(beh_mat_no_diag[:, ind]))
            
            else:
                imrates.append(np.nan)

In [ ]:
fig, ax = plt.subplots()
ax.plot(im_83_hit_rates['Familiar'], im_83_hit_rates['Novel'], 'ko')
ax.plot(im_111_hit_rates['Familiar'], im_111_hit_rates['Novel'], 'ro')
ax.set_aspect('equal')
ax.legend(['im083', 'im111'])
ax.plot([0,1], [0,1], 'k--')
ax.set_xlabel('Hit rate during Familiar session')
ax.set_ylabel('Hit rate during Novel session')



In [ ]:
plt.figure()
plt.imshow(np.nanmean(beh_mat_no_diag_array[(good_behavior_session_filter) & (experience_labels=='Novel')], axis=0))
plt.colorbar()
plt.figure()
plt.imshow(np.nanmean(beh_mat_array[(good_behavior_session_filter) & (experience_labels=='Novel')], axis=0))


In [ ]:
shared_to_shared_hit_rate = beh_mat_no_diag_array[(good_behavior_session_filter) & (experience_labels=='Novel')][:, 6, 6:]
nonshared_to_shared_hit_rate = beh_mat_no_diag_array[(good_behavior_session_filter) & (experience_labels=='Novel')][:, :6, 6:]

In [ ]:
null_shared_to_shared = []
null_nonshared_to_shared = []
for iteration in range(100):
    # Shuffle rows of each beh matrix
    #beh_mat_no_diag_array_shuff = np.array([np.random.permutation(b.flatten()).reshape(b.shape) for b in beh_mat_no_diag_array[(good_behavior_session_filter) & (experience_labels=='Novel')]])
    beh_mat_no_diag_array_shuff = np.array([np.random.permutation(b) for b in beh_mat_no_diag_array[(good_behavior_session_filter) & (experience_labels=='Novel')]])
    stos = np.nanmean(beh_mat_no_diag_array_shuff[:, 6, 6:], axis=1)
    ntos = np.nanmean(beh_mat_no_diag_array_shuff[:, :6, 6:], axis=(1,2))
    # if all([~np.isnan(val) for val in [stos, ntos]]):
    null_shared_to_shared.extend(np.nanmean(beh_mat_no_diag_array_shuff[:, 6, 6:], axis=1))
    null_nonshared_to_shared.extend(np.nanmean(beh_mat_no_diag_array_shuff[:, :6, 6:], axis=(1,2)))

null_shared_to_shared = np.array(null_shared_to_shared)
null_nonshared_to_shared = np.array(null_nonshared_to_shared)


In [ ]:
fig, ax = plt.subplots()
#ax.hist2d(null_shared_to_shared[~np.isnan(null_shared_to_shared)], null_nonshared_to_shared[~np.isnan(null_shared_to_shared)], (25,25), cmap='plasma')
ax.plot(null_shared_to_shared, null_nonshared_to_shared, 'ko', alpha=0.002)
ax.plot(np.mean(shared_to_shared_hit_rate, axis=1), np.mean(nonshared_to_shared_hit_rate, axis=(1,2)), 'ro')
# ax.plot(np.nanmean(shared_to_shared_hit_rate), np.nanmean(nonshared_to_shared_hit_rate), 'rP', ms=20, mfc='w')
# ax.plot(np.nanmean(null_shared_to_shared), np.nanmean(null_nonshared_to_shared), 'kP', ms=10)
ax.errorbar(np.nanmean(null_shared_to_shared), np.nanmean(null_nonshared_to_shared), 
                        xerr=np.nanstd(null_shared_to_shared), yerr=np.nanstd(null_nonshared_to_shared), color='k')
ax.errorbar(np.nanmean(shared_to_shared_hit_rate), np.nanmean(nonshared_to_shared_hit_rate), 
                        xerr=np.nanstd(shared_to_shared_hit_rate), yerr=np.nanstd(nonshared_to_shared_hit_rate), color='r')

ax.set_aspect('equal')
ax.plot([0,1], [0,1], 'k--')
ax.set_xlabel('From shared image')
ax.set_ylabel('From non-shared image')

scipy.stats.ranksums(null_shared_to_shared - null_nonshared_to_shared, np.mean(shared_to_shared_hit_rate, axis=1) - np.mean(nonshared_to_shared_hit_rate, axis=(1,2)), nan_policy='omit')


## Hit rates for non-omissions, omissions and post-omissions

A few notes about the stim_table columns:
- a lick bout is defined as any lick that follows the last lick by > 0.5 seconds
- `lick_time` is really 'lick bout time'. Not all licks are registered in this column, just the first licks in lick bouts. If you want to see all the licks in a trial, `lick_times` stores them in a (string) list
- `lick_for_flash` indicates whether a lick bout initiated after the start time of a flash and before the next flash. In the next cell we will add `lick_for_flash_during_response_window`, which will indicate whether a lick bout started in the response window after a flash (100-750 ms after flash start)
- `first_lick_in_trial` should be named 'first lick bout in trial'. There are some edge cases when consummatory licks from the last trial continue on into the next trial, but this trial doesn't abort immediately. In these cases, you can have a lick bout that starts after these consummatory licks, and `first_lick_in_trial` will be True, but `before_first_trial_lick` will be false (since consummatory licks happened earlier in the trial).



In [ ]:
response_rate_summary = {s:{col: np.nan for col in ['G_Familiar_private_nonchange', 'G_Familiar_shared_nonchange', 
                                                    'G_Novel_private_nonchange', 'G_Novel_shared_nonchange',
                                                    'H_Familiar_private_nonchange', 'H_Familiar_shared_nonchange', 
                                                    'H_Novel_private_nonchange', 'H_Novel_shared_nonchange',
                                                    'G_Familiar_private_hit', 'G_Familiar_shared_hit', 
                                                    'G_Novel_private_hit', 'G_Novel_shared_hit',
                                                    'H_Familiar_private_hit', 'H_Familiar_shared_hit', 
                                                    'H_Novel_private_hit', 'H_Novel_shared_hit',
                                                    'G_Familiar_private_fa', 'G_Familiar_shared_fa', 
                                                    'G_Novel_private_fa', 'G_Novel_shared_fa',
                                                    'H_Familiar_private_fa', 'H_Familiar_shared_fa', 
                                                    'H_Novel_private_fa', 'H_Novel_shared_fa',
                                                    'G_Familiar_omission', 'G_Novel_omission', 
                                                    'H_Familiar_omission', 'H_Novel_omission', 
                                                    'G_Familiar_postomission', 'G_Novel_postomission', 
                                                    'H_Familiar_postomission', 'H_Novel_postomission']} for s in good_behavior_sessions['ecephys_session_id'].values}

for isess, session in good_behavior_sessions.iterrows():
    
    experience_level = session['experience_level']
    image_set = session['image_set']
    session_id = session['ecephys_session_id']
    session_stim_table = stim_table[stim_table['session_id']==session_id]
    session_trials = session_stim_table.groupby('behavior_trial_id').head(1)

    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'private_nonchange'] = get_private_nonchange_response_rate(session_stim_table)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'private_hit'] = get_private_hit_rate(session_trials)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'private_fa'] = get_private_fa_rate(session_trials)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'shared_nonchange'] = get_shared_nonchange_response_rate(session_stim_table)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'shared_hit'] = get_shared_hit_rate(session_trials)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'shared_fa'] = get_shared_fa_rate(session_trials)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'omission'] = get_omission_response_rate(session_stim_table)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'postomission'] = get_post_omission_response_rate(session_stim_table)

In [ ]:
response_rate_df = pd.DataFrame.from_dict(response_rate_summary, orient='index')

In [ ]:
means = response_rate_df.mean()
sems = response_rate_df.sem()

fig, ax = plt.subplots()
fig.set_size_inches(12,6)
ax.errorbar(np.arange(len(means)), means, yerr=sems)
ax.set_xticks(np.arange(len(means)))
ax.set_xticklabels(means.index.values, rotation=90)

In [ ]:
aggregated_labels = ['Familiar_shared', 'Familiar_private', 'Novel_shared', 'Novel_private', 'Novel', 'Familiar']
response_categories = ['nonchange', 'hit',]# 'fa', 'omission', 'postomission']

cols_to_aggregate = []
for resp in response_categories:
    for ag_label in aggregated_labels:

        cols = [c for c in response_rate_df.columns if ag_label + '_' + resp in c]
        if len(cols)>0:
            cols_to_aggregate.append(cols)

In [ ]:
labels = []
figure_labels = []
values = []
means = []
sems = []
for cols in cols_to_aggregate:
    figure_labels.append(cols[0][2:].replace('shared', 'session \n holdover').replace('private', '').replace('_', ' ').replace('hit', 'change'))

    labels.append(cols[0][2:])
    vals = response_rate_df[cols].to_numpy().flatten()
    vals = vals[~np.isnan(vals)]

    values.append(vals)
    means.append(np.mean(vals))
    sems.append(np.std(vals)/len(vals)**0.5)

In [ ]:
fig, ax = plt.subplots()
ax.violinplot(values)
# ax.plot(means, 'ko')
# ax.errorbar(np.arange(len(means)), means, yerr=sems, color='k')
ax.set_xticks(np.arange(len(means))+1)
ax.set_xticklabels(figure_labels, rotation=90)

In [ ]:
plt.rcParams['font.size'] = 16
fig, ax = plt.subplots()
for ivals, vals in enumerate(values):
    color = 'b' if 'Familiar' in labels[ivals] else 'r'
    ax.boxplot(vals, positions=[ivals+1], widths=0.5, showfliers=False, whis=(10, 90), notch=True, boxprops={'color': color, 'linewidth': 2}, medianprops={'color': color, 'linewidth': 2}, whiskerprops={'color': color, 'linewidth': 2}, capprops={'color': color, 'linewidth': 2})

ax.set_xticklabels(figure_labels, rotation=90)
vbn_utils.formatFigure(fig, ax, yLabel='Response rate')
fig.savefig("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/behavioral_response_rate_boxplot_test.png")


## Statistical comparisons of response rates

In [ ]:
vs, ps = vbn_utils.comparison_matrix(*values)
vs[1, 0]

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(7,7)
vbn_utils.plot_comparison_matrix(*values, colorbar=True, ax=ax, cmap='PiYG', binarize=True)
ax.set_xticklabels(figure_labels, rotation=90)
ax.set_yticklabels(figure_labels,)

## Reaction time analysis

In [ ]:
## Reaction time stuff

In [ ]:
stim_table['shared'] = stim_table['image_name'].isin(['im083_r', 'im111_r'])

In [ ]:
conditions = ['private_nonchange', 'shared_nonchange', 
                'private_change', 'shared_change',
                'omitted', 'previous_omitted']
reaction_time_summary = {s:{} for s in good_behavior_sessions['ecephys_session_id'].values}

for isess, session in good_behavior_sessions.iterrows():
    
    experience_level = session['experience_level']
    image_set = session['image_set']
    session_id = session['ecephys_session_id']
    session_stim_table = stim_table[stim_table['session_id']==session_id]
    session_trials = session_stim_table.groupby('behavior_trial_id').head(1)

    reaction_time_summary[session_id]['image_set'] = image_set
    reaction_time_summary[session_id]['experience_level'] = experience_level
    reaction_time_summary[session_id]['mouse_id'] = session['mouse_id']

    for condition in conditions:
        if 'private' in condition or 'shared' in condition:
            shared_or_private_query = 'shared' if 'shared' in condition else '~shared'
        else:
            shared_or_private_query = 'trial_flash>-1' #always true
        change_or_nonchange_query = 'is_change' if '_change' in condition else '~is_change'
        omission_query = 'omitted' if condition=='omitted' else '~omitted'
        previous_omission_query = 'previous_omitted' if condition=='previous_omitted' else '~previous_omitted'
        
        reaction_time_summary[session_id][condition] = calculate_metric_for_selection(session_stim_table, 'reaction_time', np.nanmean, 
                                                                                        shared_or_private_query, 
                                                                                        change_or_nonchange_query, 
                                                                                        omission_query, 
                                                                                        previous_omission_query,
                                                                                        'engaged',
                                                                                        'lickbout_for_flash_during_response_window',
                                                                                        'no_abnorm',
                                                                                        'change_eligible_window') 

In [ ]:
reaction_time_df = pd.DataFrame.from_dict(reaction_time_summary, orient='index')

In [ ]:
mouse_session_counts = good_behavior_sessions.value_counts('mouse_id')
mice_with_fam_and_nov_sessions = mouse_session_counts[mouse_session_counts > 1].index.values

g_familiar_mice = reaction_time_df[(reaction_time_df['experience_level']=='Familiar') & (reaction_time_df['image_set']=='G')]['mouse_id'].unique()
h_familiar_mice = reaction_time_df[(reaction_time_df['experience_level']=='Familiar') & (reaction_time_df['image_set']=='H')]['mouse_id'].unique()

paired_reaction_time_df = reaction_time_df[reaction_time_df['mouse_id'].isin(list(mice_with_fam_and_nov_sessions))]
paired_reaction_time_df

In [ ]:
paired_rt_diff_novel = []
paired_rt_diff_familiar = []
for mouse in mice_with_fam_and_nov_sessions:
    for experience, rt_diffs in zip(['Novel', 'Familiar'], [paired_rt_diff_novel, paired_rt_diff_familiar]): 
        mouse_df = paired_reaction_time_df[(paired_reaction_time_df['mouse_id']==mouse)&(paired_reaction_time_df['experience_level']==experience)]
        rt_diff = mouse_df['shared_change'].values - mouse_df['private_change'].values
        rt_diffs.append(rt_diff[0])

fig, ax = plt.subplots()
# ax.bar([0,1], [np.mean(paired_rt_diff_novel), np.mean(paired_rt_diff_familiar)], yerr=[np.std(paired_rt_diff_novel), np.std(paired_rt_diff_familiar)])
for nov, fam in zip(paired_rt_diff_novel, paired_rt_diff_familiar):
    ax.plot([0,1], [nov, fam], 'k-o')


In [ ]:
scipy.stats.wilcoxon(paired_rt_diff_novel, paired_rt_diff_familiar)

In [ ]:
behavior_model_summary = pd.read_pickle("/Volumes/programs/braintv/workgroups/nc-ophys/alex.piet/NP/behavior/psy_fits_v100/summary_data/_summary_table.pkl")
timing_sessions = behavior_model_summary[behavior_model_summary['strategy_labels']=='timing'].index.values
visual_sessions = behavior_model_summary[behavior_model_summary['strategy_labels']=='visual'].index.values

In [ ]:
reaction_time_df['timing'] = False
reaction_time_df.loc[[t for t in timing_sessions if t in reaction_time_df.index.values], 'timing'] = True

In [ ]:
reaction_time_df.pivot_table(index=['experience_level', 'image_set'], values=conditions, aggfunc=(np.nanmedian, 'count'))

In [ ]:

_, count_mat, mean_mat = paired_image_mat_from_stim_table(stim_table, 'reaction_time', 'G', np.nanmean, 'is_change', 'engaged', 'no_abnorm', 'experience_level=="Novel"', 'lickbout_for_flash_during_response_window')

In [ ]:
_, count_mat, mean_mat = paired_image_mat_from_stim_table(stim_table, 'reaction_time', 'H', np.nanmean, 'is_change', 'engaged', 'no_abnorm', 'experience_level=="Novel"', 'lickbout_for_flash_during_response_window')
plt.figure()
plt.imshow(mean_mat)
plt.colorbar()

_, count_mat, mean_mat = paired_image_mat_from_stim_table(stim_table, 'reaction_time', 'H', np.nanmean, 'is_change', 'engaged', 'no_abnorm', 'experience_level=="Familiar"', 'lickbout_for_flash_during_response_window')

plt.figure()
plt.imshow(mean_mat)
plt.colorbar()

In [ ]:
rt_values = []
rt_labels = []
for experience_level in ['Familiar', 'Novel']:
    for image_set in ['G', 'H']:
        for shared_or_private in ['shared', 'private']:
            for change_or_nonchange in ['change','nonchange']:
            
                shared_filter = stim_table['shared'] if shared_or_private=='shared' else ~stim_table['shared']
                change_filter = stim_table['is_change'] if change_or_nonchange=='change' else ~stim_table['is_change']
                good_behavior_filter = stim_table['session_id'].isin(good_behavior_sessions['ecephys_session_id'].values)
                stim_subset = stim_table[(stim_table['experience_level']==experience_level)&
                                        (stim_table['image_set']==image_set)&
                                        (stim_table['engaged'])&
                                        (stim_table['no_abnorm'])&
                                        (stim_table['lickbout_for_flash_during_response_window'])&
                                        (stim_table['change_eligible_window'])&
                                        (shared_filter) & (change_filter) & good_behavior_filter]
                rt_values.append(stim_subset['reaction_time'].values)
                rt_labels.append(f'{experience_level}_{image_set}_{shared_or_private}_{change_or_nonchange}')

In [ ]:
pvals, sigs = pvalue_comparison_matrix(*rt_values, test_func=scipy.stats.ranksums)

In [ ]:
fig, ax = plt.subplots()
# Plot the modified array
plt.xticks(range(len(pvals)), list(rt_labels), rotation=90)
plt.yticks(range(len(pvals)), list(rt_labels))
# Draw red asterisks at coordinates where pvals < 0.05
ax.plot(*np.where(sigs), 'w*')
im = ax.imshow(pvals, clim=(0,1))
plt.colorbar(im, label='corrected p-value')

In [ ]:
rt_dict = {label: val for label, val in zip(rt_labels, rt_values)}
plt.figure()
plt.hist(rt_dict['Novel_H_private_nonchange'], alpha=0.5, density=True)
plt.hist(rt_dict['Novel_H_private_change'], alpha=0.5, density=True)

print(np.nanmean(rt_dict['Novel_H_private_nonchange']), np.nanmean(rt_dict['Novel_H_private_change']))

In [ ]:
rt_values = []
rt_labels = []
for experience_level in ['Familiar', 'Novel']:
    for image_set in ['G', 'H']:
        if experience_level == 'Familiar' and image_set == 'H':
                continue
        if experience_level == 'Novel' and image_set == 'G':
                continue
        for cols in ['private_change', 'shared_change', 'shared_nonchange']:

            rt_values.append(paired_reaction_time_df[(paired_reaction_time_df['experience_level']==experience_level)&(paired_reaction_time_df['image_set']==image_set)][cols].values)
            rt_labels.append(f'{experience_level}_{image_set}_{cols}')

In [ ]:
fams = paired_reaction_time_df[(paired_reaction_time_df['experience_level']=='Familiar')]# & (paired_reaction_time_df['image_set']=='G')]
novs = paired_reaction_time_df[(paired_reaction_time_df['experience_level']=='Novel')]# & (paired_reaction_time_df['image_set']=='H')]

private_fam = fams.sort_values(by='mouse_id')['private_change']
private_nov = novs.sort_values(by='mouse_id')['private_change']

scipy.stats.wilcoxon(private_fam, private_nov, nan_policy='omit'), np.mean(private_fam), np.mean(private_nov), len(private_fam), len(private_nov)

In [ ]:
plt.figure()
plt.plot(private_fam, private_nov, 'ko')
plt.plot([0,1], [0,1], 'k--')
plt.xlim(0.2,0.6)
plt.ylim(0.2,0.6)

plt.figure()
plt.boxplot(private_fam.values - private_nov.values, labels=['diff'], showfliers=False)
plt.gca().axhline(0, color='k', ls='--')

In [ ]:
pvals, sigs = pvalue_comparison_matrix(*rt_values, test_func=scipy.stats.wilcoxon)

In [ ]:
fig, ax = plt.subplots()
# Plot the modified array
plt.xticks(range(len(pvals)), list(rt_labels), rotation=90)
plt.yticks(range(len(pvals)), list(rt_labels))
# Draw red asterisks at coordinates where pvals < 0.05
ax.plot(*np.where(sigs), 'w*')
im = ax.imshow(pvals, clim=(0,1))
plt.colorbar(im, label='corrected p-value')

## Running speed during active and passive contexts

In [ ]:
passing_sessions = sessions_table[sessions_table['abnormal_activity'].isnull() & sessions_table['abnormal_histology'].isnull()]['ecephys_session_id'].values

familiar_running = stim_table[stim_table['engaged'] & (stim_table['experience_level']=='Familiar') & (~stim_table['is_change']) & (stim_table['hit']) &
                              (stim_table['session_id'].isin(passing_sessions)) & (~stim_table['image_name'].isin(['im083_r', 'im111_r']))]['active_baseline_running_speed']
novel_running = stim_table[stim_table['engaged'] & (stim_table['experience_level']=='Novel') & (~stim_table['is_change']) & (stim_table['hit']) &
                           (stim_table['session_id'].isin(passing_sessions)) & (~stim_table['image_name'].isin(['im083_r', 'im111_r']))]['active_baseline_running_speed']

plt.hist(familiar_running[~np.isnan(familiar_running)], density=True, alpha=0.5, bins=np.arange(0, 150, 5), color='b')
plt.hist(novel_running[~np.isnan(novel_running)], density=True, alpha=0.5, bins=np.arange(0, 150, 5), color='r')

print(scipy.stats.ranksums(familiar_running[~np.isnan(familiar_running)], novel_running[~np.isnan(novel_running)]))
print(np.nanmedian(familiar_running), np.nanmedian(novel_running))

In [ ]:
active_running = stim_table[stim_table['engaged'] & (stim_table['session_id'].isin(passing_sessions)) & (~stim_table['is_change']) & (stim_table['hit'])]['active_baseline_running_speed']
passive_running = stim_table[stim_table['engaged'] & (stim_table['session_id'].isin(passing_sessions)) & (~stim_table['is_change']) & (stim_table['hit'])]['passive_baseline_running_speed']

plt.hist(active_running[~np.isnan(active_running)], density=True, alpha=0.5, bins=np.arange(0, 150, 5), color='k')
plt.hist(passive_running[~np.isnan(passive_running)], density=True, alpha=0.5, bins=np.arange(0, 150, 5), color='g')

scipy.stats.ranksums(active_running[~np.isnan(active_running)], passive_running[~np.isnan(passive_running)])